In [ ]:
# ============================================================
# 5.1 Mixture Density Network – full solution code
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_probability as tfp

tfd = tfp.distributions

# ------------------------------------------------------------
# a) Generate samples for A1, A2
# ------------------------------------------------------------

np.random.seed(0)
tf.random.set_seed(0)

n_samples = 1000
A_min_true = 40.0   # mm^2
A_max_true = 160.0  # mm^2

A1 = np.random.uniform(A_min_true, A_max_true, size=(n_samples, 1))
A2 = np.random.uniform(A_min_true, A_max_true, size=(n_samples, 1))

# ------------------------------------------------------------
# Helper: mechanical model u(F,l,E,A,alpha)
# u = F*l*(1 + cos^2(alpha)) / (E*A*sin^2(alpha)*cos(alpha))
# ------------------------------------------------------------

def displacement(F, l, E, A, alpha):
    num = F * l * (1.0 + np.cos(alpha) ** 2)
    den = E * A * (np.sin(alpha) ** 2) * np.cos(alpha)
    return num / den

# ------------------------------------------------------------
# b) Create u1, u2 with noise
# ------------------------------------------------------------

sigma_n = 0.05
E1 = 210000.0  # MPa
E2 = 100000.0  # MPa
l = 1000.0     # mm
alpha = np.pi / 4.0
F = 1000.0     # N

u1 = displacement(F, l, E1, A1, alpha)
u2 = displacement(F, l, E2, A2, alpha)

u1 += np.random.normal(0.0, sigma_n, size=u1.shape)
u2 += np.random.normal(0.0, sigma_n, size=u2.shape)

# ------------------------------------------------------------
# c) Concatenate to one input vector (A) and one output vector (u)
# ------------------------------------------------------------

A_all = np.vstack([A1, A2])           # shape (2000, 1)
u_all = np.vstack([u1, u2])           # shape (2000, 1)

# ------------------------------------------------------------
# d) Scale both vectors to [0, 1]
# ------------------------------------------------------------

A_min, A_max = A_all.min(), A_all.max()
u_min, u_max = u_all.min(), u_all.max()

A_scaled = (A_all - A_min) / (A_max - A_min)
u_scaled = (u_all - u_min) / (u_max - u_min)

# ------------------------------------------------------------
# e) Build Mixture Density Network
# ------------------------------------------------------------

n_components = 2  # mixture of 2 Gaussians

inputs = tf.keras.Input(shape=(1,), name="A_scaled")

h = tf.keras.layers.Dense(500, activation="relu")(inputs)
h = tf.keras.layers.Dense(500, activation="relu")(h)

# Three output heads: means, sigmas, mixing probs
means = tf.keras.layers.Dense(n_components, name="means")(h)
sigmas = tf.keras.layers.Dense(
    n_components,
    activation=tf.exp,     # exponential to enforce positivity
    name="sigmas",
)(h)
pis = tf.keras.layers.Dense(
    n_components,
    activation="softmax",  # mixture probabilities
    name="pis",
)(h)

# Final MDN output concatenates [means, sigmas, pis]
outputs = tf.keras.layers.Concatenate(name="mdn_output")([means, sigmas, pis])

mdn_model = tf.keras.Model(inputs=inputs, outputs=outputs, name="MDN")

# ------------------------------------------------------------
# f) Custom MDN loss (negative log-likelihood)
# ------------------------------------------------------------

def mdn_loss(y_true, y_pred):
    """
    y_true: (batch, 1) – scaled displacement
    y_pred: (batch, 3 * n_components) – [means, sigmas, pis]
    """
    y_true = tf.squeeze(y_true, axis=-1)  # (batch,)

    means_pred = y_pred[:, :n_components]
    sigmas_pred = y_pred[:, n_components:2 * n_components]
    pis_pred = y_pred[:, 2 * n_components:]

    # Mixture of Gaussians p(y | x)
    mixture = tfd.MixtureSameFamily(
        mixture_distribution=tfd.Categorical(probs=pis_pred),
        components_distribution=tfd.Normal(
            loc=means_pred,
            scale=sigmas_pred,
        ),
    )

    log_prob = mixture.log_prob(y_true)  # (batch,)
    # The exercise text asks for the negative SUM of log-likelihoods:
    return -tf.reduce_sum(log_prob)

# ------------------------------------------------------------
# g) Compile and train
# ------------------------------------------------------------

mdn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=mdn_loss,
)

history = mdn_model.fit(
    A_scaled,
    u_scaled,
    epochs=100,
    batch_size=64,
    verbose=1,
    shuffle=True,
)

# ------------------------------------------------------------
# h) Plot training data, predicted means and 95%-intervals
# ------------------------------------------------------------

# Predictions on the training inputs
y_pred = mdn_model.predict(A_scaled, batch_size=256)
means_pred = y_pred[:, :n_components]            # (2000, 2)
sigmas_pred = y_pred[:, n_components:2*n_components]

# Undo scaling for plotting in original units
def unscale_u(u_scaled):
    return u_scaled * (u_max - u_min) + u_min

u_train = u_all.ravel()
A_train = A_all.ravel()

means_phys = unscale_u(means_pred)
sigmas_phys = sigmas_pred * (u_max - u_min)

# Sort by A so curves look nice
idx = np.argsort(A_train)
A_sorted = A_train[idx]
u_sorted = u_train[idx]
mu1 = means_phys[idx, 0]
mu2 = means_phys[idx, 1]
s1 = sigmas_phys[idx, 0]
s2 = sigmas_phys[idx, 1]

plt.figure(figsize=(8, 6))
plt.scatter(A_train, u_train, s=10, alpha=0.3, label="training data")

plt.plot(A_sorted, mu1, label=r"$\mu_1$")
plt.plot(A_sorted, mu2, label=r"$\mu_2$")

plt.fill_between(
    A_sorted,
    mu1 - 1.96 * s1,
    mu1 + 1.96 * s1,
    alpha=0.2,
    label=r"$\mu_1 \pm 1.96\sigma_1$",
)
plt.fill_between(
    A_sorted,
    mu2 - 1.96 * s2,
    mu2 + 1.96 * s2,
    alpha=0.2,
    label=r"$\mu_2 \pm 1.96\sigma_2$",
)

plt.xlabel(r"Cross section $A$ [$\mathrm{mm}^2$]")
plt.ylabel("Displacement u")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


: 